# Transformer baseline for agricultural insurance loss prediction

This notebook implements a compact Transformer-based baseline that mirrors the XGBoost notebooks data loading, temporal split, and evaluation. It trains a small Transformer (joint classification + regression heads) on sequence windows and saves predictions to the processed folder for easy comparison with the XGBoost outputs.

Notes:
- Uses seq_len=12 by default (adjustable).
- Trains for a small number of epochs for a quick smoke test; increase for full experiments.
- Saves predictions to `data/processed/model_predictions_transformer.csv`.

In [48]:
# Imports and device
import os
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import DataLoader, TensorDataset

In [49]:
def create_windows(df, feature_cols, seq_len=12, group_col='location'):
    """Produce sliding windows per-group.
    Returns: X (n, seq_len, n_features), y (n,), y_bin (n,), meta (DataFrame)
    """
    windows = []
    metas = []

    # Choose a valid grouping column
    if group_col not in df.columns:
        group_col = 'county_state' if 'county_state' in df.columns else df.columns[0]

    # Determine which meta columns are available to avoid KeyError
    candidate_meta_cols = ['location', 'county_state', 'year', 'month', 'commodity_year', 'date', 'join_key']
    available_meta_cols = [c for c in candidate_meta_cols if c in df.columns]

    grouped = df.sort_values(['year', 'month']).groupby(group_col)
    for key, g in grouped:
        arr = g[feature_cols].fillna(0).values.astype(np.float32)
        losses = g['loss'].fillna(0).values.astype(np.float32)
        n = len(g)
        if n <= seq_len:
            continue
        for end in range(seq_len, n):
            start = end - seq_len
            Xw = arr[start:end]  # (seq_len, n_features)
            y = losses[end]
            yb = 1.0 if y > 0 else 0.0
            # Build meta row using only available meta columns (avoid KeyError if 'date' or others missing)
            if len(available_meta_cols) > 0:
                meta_row = g.iloc[end][available_meta_cols].to_dict()
            else:
                # fallback: include group key and index
                meta_row = {group_col: key, 'index': int(g.index[end])}
            windows.append((Xw, y, yb, meta_row))

    if len(windows) == 0:
        return np.zeros((0, seq_len, len(feature_cols)), dtype=np.float32), np.zeros((0,)), np.zeros((0,)), pd.DataFrame()

    X = np.stack([w[0] for w in windows], axis=0)
    y = np.array([w[1] for w in windows], dtype=np.float32)
    yb = np.array([w[2] for w in windows], dtype=np.float32)
    meta = pd.DataFrame([w[3] for w in windows])
    return X, y, yb, meta

In [50]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x):
        # x: (batch, seq_len, d_model) expected by our usage, but positional buffer is (max_len,1,d_model)
        x = x + self.pe[:x.size(1)].permute(1,0,2).squeeze(1) if False else x
        # We'll add positional enc via standard sin/cos using alternative approach below in model to keep shapes clear
        return self.dropout(x)
class TimeSeriesTransformer(nn.Module):
    def __init__(self, n_num, d_model=64, n_heads=4, n_layers=2, dim_ff=128, dropout=0.1, n_static=0):
        super().__init__()
        self.num_proj = nn.Linear(n_num, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, 500, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=dim_ff, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.n_static = n_static
        total_rep = d_model * 12 + n_static if False else d_model  # we'll pool instead of flattening full sequence
        # Use pooled representation
        self.classifier = nn.Sequential(nn.Linear(d_model + n_static, d_model//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_model//2, 1))
        self.regressor = nn.Sequential(nn.Linear(d_model + n_static, d_model//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_model//2, 1))
    def forward(self, x_seq, x_static=None, src_key_padding_mask=None):
        # x_seq: (batch, seq_len, n_num)
        z = self.num_proj(x_seq)  # (b, seq, d)
        seq_len = z.size(1)
        pos = self.pos_emb[:, :seq_len, :]
        z = z + pos
        z = self.transformer(z, src_key_padding_mask=src_key_padding_mask)  # (b, seq, d)

        z_pool = z.mean(dim=1)  # (b, d)
        if x_static is not None:
            rep = torch.cat([z_pool, x_static], dim=1)
        else:
            rep = z_pool
        logits = self.classifier(rep).squeeze(-1)
        prob = torch.sigmoid(logits)
        reg = torch.relu(self.regressor(rep).squeeze(-1))
        return prob, reg, logits

In [51]:
def train_epoch(model, loader, optimizer, device, alpha=1.0):
    """Single-stage training epoch. Regression target is trained in log-space (log1p) when positive."""
    model.train()
    total_loss = 0.0
    bce_loss = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()
    for X, Xs, y, yb in loader:
        X = X.to(device)
        y = y.to(device)
        yb = yb.to(device)
        if Xs is not None:
            Xs = Xs.to(device)
        optimizer.zero_grad()
        prob, reg_out, logits = model(X, Xs)
        # logits is already available; compute bce on logits vs yb
        loss_clf = bce_loss(logits, yb)
        mask_pos = (yb > 0.5)
        if mask_pos.sum() > 0:
            # train regressor to predict log1p(loss) for positive samples
            target_log = torch.log1p(y[mask_pos])
            loss_reg = mse(reg_out[mask_pos], target_log)
        else:
            loss_reg = torch.tensor(0.0, device=device)
        loss = loss_clf + alpha * loss_reg
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
    return total_loss / len(loader.dataset)
def evaluate(model, X, Xs, y, device, threshold=0.5):
    """Evaluate model. Regressor predicts log1p(loss) — backtransform with expm1 for final predicted_loss."""
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        Xs_t = torch.tensor(Xs, dtype=torch.float32).to(device) if Xs is not None else None
        prob, reg_out, logits = model(X_t, Xs_t)
        prob = prob.cpu().numpy()
        reg_out = reg_out.cpu().numpy()  # this is log1p(pred) in numpy
        # back-transform regressed log-space outputs to original loss scale
        regs_back = np.expm1(reg_out)
        regs_back = np.maximum(regs_back, 0.0)
        preds = (prob > threshold).astype(float) * regs_back
        return preds, prob, reg_out

In [52]:
def main_pipeline(seq_len=12, epochs=5, batch_size=64, lr=1e-3, device=None, two_stage=False, epochs_cls=3, epochs_reg=3, alpha=1.0):
    """Main pipeline. If two_stage=True, performs: (1) classifier-focused training for epochs_cls, (2) freeze classifier and train regressor on positive samples for epochs_reg."""
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('Using device:', device)
    aug_path = r"C:\Users\Arnold\OneDrive\Desktop\CAPSTONE PROJECT\farming_risk_regions\data\processed\merged_weather_insurance_data_with_lag.csv"
    base_path = r"C:\Users\Arnold\OneDrive\Desktop\CAPSTONE PROJECT\farming_risk_regions\data\processed\merged_weather_insurance_data.csv"
    path = aug_path if os.path.exists(aug_path) else base_path
    df = pd.read_csv(path)
    print(f'Loaded {len(df)} rows from {path}')
    # Exclude leaking columns similar to XGBoost notebook
    exclude = ['loss','loss_ratio','year','month','date','join_key','location','county_state','commodity_year']
    feature_cols = [c for c in df.columns if c not in exclude]
    print('Using features sample:', feature_cols[:10])
    # Temporal train/test split 80/20 as in XGBoost notebook
    df = df.sort_values(['year','month']).reset_index(drop=True)
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx].reset_index(drop=True)
    test_df = df.iloc[split_idx:].reset_index(drop=True)
    # Create windows
    X_train, y_train, yb_train, meta_train = create_windows(train_df, feature_cols, seq_len=seq_len)
    X_test, y_test, yb_test, meta_test = create_windows(test_df, feature_cols, seq_len=seq_len)
    print(f'Windows: train {len(X_train)}, test {len(X_test)}')
    if len(X_train) == 0 or len(X_test) == 0:
        raise RuntimeError('Not enough windowed data for seq_len=' + str(seq_len))
    # scale numeric features by train mean/std per feature (across time and windows)
    mean = X_train.mean(axis=(0,1), keepdims=True)
    std = X_train.std(axis=(0,1), keepdims=True) + 1e-6
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    # No static features used for now (placeholder zeros)
    Xs_train = np.zeros((len(X_train), 0), dtype=np.float32)
    Xs_test = np.zeros((len(X_test), 0), dtype=np.float32)
    # Build dataloaders
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(Xs_train), torch.tensor(y_train), torch.tensor(yb_train))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    # Model
    model = TimeSeriesTransformer(n_num=X_train.shape[2], d_model=64, n_heads=4, n_layers=2, dim_ff=128, n_static=0).to(device)
    # Optimizer will be (re)created depending on two-stage training
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # Two-stage training: (1) classifier-focused, (2) regressor on positives
    if two_stage:
        print('Running two-stage training: classifier for', epochs_cls, 'epochs; regressor for', epochs_reg, 'epochs')
        # Phase 1: classifier-only (optimize BCE on logits)
        bce = nn.BCEWithLogitsLoss()
        for ep in range(1, epochs_cls+1):
            model.train()
            total = 0.0
            for Xb, Xsb, yb_vals, yb_flags in train_loader:
                Xb = Xb.to(device)
                yb_flags = yb_flags.to(device)
                if Xsb is not None and Xsb.nelement() > 0:
                    Xsb = Xsb.to(device)
                optimizer.zero_grad()
                _, _, logits = model(Xb, Xsb)
                loss = bce(logits, yb_flags)
                loss.backward()
                optimizer.step()
                total += loss.item() * Xb.size(0)
            print(f'Classifier epoch {ep}/{epochs_cls} - loss:', total / len(train_loader.dataset))
        # Phase 2: freeze classifier and train regressor on positive examples only
        for p in model.classifier.parameters():
            p.requires_grad = False
        # Recreate optimizer for trainable params (regressor + transformer)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
        mse = nn.MSELoss()
        # Build positive-only dataset
        pos_mask = (yb_train > 0.5)
        if pos_mask.sum() == 0:
            print('No positive samples in training data; skipping regressor training')
        else:
            X_pos = X_train[pos_mask]
            y_pos = y_train[pos_mask]
            Xs_pos = Xs_train[pos_mask] if Xs_train.shape[1] > 0 else np.zeros((len(X_pos),0), dtype=np.float32)
            # train regressor on log1p(y) to stabilize heavy tails
            pos_ds = TensorDataset(torch.tensor(X_pos), torch.tensor(Xs_pos), torch.tensor(np.log1p(y_pos)))
            pos_loader = DataLoader(pos_ds, batch_size=batch_size, shuffle=True)
            for ep in range(1, epochs_reg+1):
                model.train()
                total_r = 0.0
                for Xb, Xsb, yb_vals in pos_loader:
                    Xb = Xb.to(device)
                    yb_vals = yb_vals.to(device)
                    if Xsb is not None and Xsb.nelement() > 0:
                        Xsb = Xsb.to(device)
                    optimizer.zero_grad()
                    _, reg_out, _ = model(Xb, Xsb)
                    loss_r = mse(reg_out, yb_vals)
                    loss_r.backward()
                    optimizer.step()
                    total_r += loss_r.item() * Xb.size(0)
                print(f'Regressor epoch {ep}/{epochs_reg} - mse:', total_r / len(pos_loader.dataset))
    else:
        # Single-stage (original joint training)
        for epoch in range(1, epochs+1):
            train_loss = train_epoch(model, train_loader, optimizer, device, alpha=alpha)
            print(f'Epoch {epoch}/{epochs} - train_loss: {train_loss:.4f}')

    # Evaluate on test set (use default threshold=0.5)
    preds, probs, regs = evaluate(model, X_test, Xs_test, y_test, device, threshold=0.5)
    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    total_actual = y_test.sum()
    total_pred = preds.sum()
    loss_ratio = total_pred / max(total_actual, 1)
    print('Test metrics: RMSE', rmse, 'MAE', mae, 'R2', r2, 'loss_ratio', loss_ratio)
    # Save predictions with meta
    meta_test = meta_test.reset_index(drop=True)
    meta_test['actual_loss'] = y_test
    meta_test['predicted_loss'] = preds
    meta_test['loss_probability'] = probs
    out_path = r'C:\Users\Arnold\OneDrive\Desktop\CAPSTONE PROJECT\farming_risk_regions\data\processed\model_predictions_transformer.csv'
    meta_test.to_csv(out_path, index=False)
    print('Saved predictions to', out_path)
    return model, dict(rmse=rmse, mae=mae, r2=r2, loss_ratio=loss_ratio), meta_test

In [53]:
# Run a quick smoke test (short epochs)
if __name__ == '__main__':
    model, metrics, preds = main_pipeline(seq_len=12, epochs=30, batch_size=128, lr=1e-3)
    print('Done. Metrics:', metrics)

Using device: cpu
Loaded 19811 rows from C:\Users\Arnold\OneDrive\Desktop\CAPSTONE PROJECT\farming_risk_regions\data\processed\merged_weather_insurance_data_with_lag.csv
Using features sample: ['ppt_total', 'rainy_days', 'ppt_variance', 'tmax', 'tmin', 'temp_range', 'tmax30', 'tmax35', 'tmin0', 'gdd']
Windows: train 15740, test 3855
Epoch 1/30 - train_loss: 1.1272
Epoch 2/30 - train_loss: 1.0741
Epoch 3/30 - train_loss: 1.0228
Epoch 4/30 - train_loss: 0.9870
Epoch 5/30 - train_loss: 0.9666
Epoch 6/30 - train_loss: 0.9469
Epoch 7/30 - train_loss: 0.9289
Epoch 8/30 - train_loss: 0.9170
Epoch 9/30 - train_loss: 0.9057
Epoch 10/30 - train_loss: 0.8995
Epoch 11/30 - train_loss: 0.8918
Epoch 12/30 - train_loss: 0.8824
Epoch 13/30 - train_loss: 0.8759
Epoch 14/30 - train_loss: 0.8635
Epoch 15/30 - train_loss: 0.8529
Epoch 16/30 - train_loss: 0.8552
Epoch 17/30 - train_loss: 0.8407
Epoch 18/30 - train_loss: 0.8304
Epoch 19/30 - train_loss: 0.8258
Epoch 20/30 - train_loss: 0.8281
Epoch 21/30 - 